# EDA: Diabetes 30-Day Readmission Prediction

이 노트북은 `diabetic_data.csv`를 사용하여 당뇨 환자의 30일 이내 재입원 여부 예측 프로젝트를 위한 탐색적 데이터 분석(EDA)을 수행합니다.

모델 학습, scaling, encoding, K-Means fitting은 수행하지 않습니다. 이 노트북의 목적은 데이터 구조, 결측치, target 분포, 주요 변수와 30일 이내 재입원 여부의 관계를 이해하고 이후 모델링 notebook의 전처리 방향을 정리하는 것입니다.

## 1. Import Libraries

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

DATA_PATH = '/content/drive/MyDrive/SKKU/M2_1/diabetic_data.csv'
FIGURE_DIR = 'figures'
os.makedirs(FIGURE_DIR, exist_ok=True)

def save_current_figure(filename):
    """현재 matplotlib figure를 figures 폴더에 저장합니다."""
    path = os.path.join(FIGURE_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved figure: {path}')

def display_rate_table(data, group_col, target_col='target_binary', top_n=None):
    """그룹별 빈도와 30일 이내 재입원률을 계산합니다."""
    table = data.groupby(group_col)[target_col].agg(['count', 'mean']).reset_index()
    table = table.rename(columns={'count': 'count', 'mean': 'readmission_rate'})
    table = table.sort_values('readmission_rate', ascending=False)
    if top_n is not None:
        table = table.head(top_n)
    return table

def plot_group_rate(data, group_col, title, filename, top_n=None, rotate=True):
    """그룹별 30일 이내 재입원률을 표와 bar plot으로 출력합니다."""
    table = display_rate_table(data, group_col, top_n=top_n)
    display(table)

    plt.figure(figsize=(10, 4))
    sns.barplot(data=table, x=group_col, y='readmission_rate', color='steelblue')
    plt.title(title)
    plt.xlabel(group_col)
    plt.ylabel('30-day readmission rate')
    if rotate:
        plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    save_current_figure(filename)
    plt.show()
    return table

## 2. Load Dataset

In [ ]:
# Google Colab에서 Google Drive를 마운트합니다.
from google.colab import drive
drive.mount('/content/drive')

# 모델링 notebook들과 같은 Google Drive 경로에서 diabetic_data.csv를 불러옵니다.
df = pd.read_csv(DATA_PATH)

print('Dataset shape:', df.shape)
display(df.head())

print('\nDataFrame info')
display(df.info())

print('\nDescriptive statistics for numeric columns')
display(df.describe())

## 3. Basic Data Overview

In [ ]:
n_rows, n_cols = df.shape
print(f'Number of rows: {n_rows:,}')
print(f'Number of columns: {n_cols:,}')

print('\nColumn names')
display(pd.Series(df.columns, name='columns'))

print('\nColumn dtypes')
dtype_summary = df.dtypes.reset_index()
dtype_summary.columns = ['column', 'dtype']
display(dtype_summary)

numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()

print('\nNumeric columns:', len(numeric_columns))
display(numeric_columns)

print('\nCategorical columns:', len(categorical_columns))
display(categorical_columns)

duplicate_rows = df.duplicated().sum()
print(f'\nDuplicate rows: {duplicate_rows:,}')

## 4. Missing Value Analysis

`diabetic_data.csv`에서는 결측치가 실제 `NaN`이 아니라 문자열 `?`로 저장된 경우가 있습니다. 따라서 EDA에서는 먼저 `?`를 `np.nan`으로 변환한 뒤 결측 비율을 확인합니다.

`weight`, `payer_code`, `medical_specialty`는 결측 비율이 높은 변수로 알려져 있으며, 모델링 단계에서 제거 후보로 고려합니다.

In [ ]:
# 원본 df는 보존하고, EDA용 복사본에서 '?'를 NaN으로 변환합니다.
eda_df = df.replace('?', np.nan).copy()

missing_value_summary = pd.DataFrame({
    'missing_count': eda_df.isna().sum(),
    'missing_ratio': eda_df.isna().mean()
}).sort_values('missing_ratio', ascending=False)

missing_value_summary['missing_percent'] = (missing_value_summary['missing_ratio'] * 100).round(2)

display(missing_value_summary)
missing_value_summary.to_csv('missing_value_summary.csv')
print('Saved missing value summary to missing_value_summary.csv')

top_missing = missing_value_summary.head(15).reset_index().rename(columns={'index': 'column'})

plt.figure(figsize=(10, 5))
sns.barplot(data=top_missing, x='missing_percent', y='column', color='salmon')
plt.title('Top 15 Columns by Missing Value Percentage')
plt.xlabel('Missing percentage (%)')
plt.ylabel('Column')
plt.tight_layout()
save_current_figure('missing_value_top15.png')
plt.show()

## 5. Target Variable Analysis

Target variable은 `readmitted`입니다. 원래 값은 `NO`, `>30`, `<30`으로 구성되어 있으며, 본 프로젝트에서는 `<30`만 30일 이내 재입원으로 정의합니다.

In [ ]:
readmitted_counts = eda_df['readmitted'].value_counts(dropna=False)
readmitted_ratio = eda_df['readmitted'].value_counts(normalize=True, dropna=False)

target_distribution = pd.DataFrame({
    'count': readmitted_counts,
    'ratio': readmitted_ratio
})
target_distribution['percent'] = (target_distribution['ratio'] * 100).round(2)

display(target_distribution)
target_distribution.to_csv('target_distribution.csv')
print('Saved target distribution to target_distribution.csv')

plt.figure(figsize=(6, 4))
sns.countplot(data=eda_df, x='readmitted', order=['NO', '>30', '<30'])
plt.title('Original readmitted Distribution')
plt.xlabel('readmitted')
plt.ylabel('Count')
plt.tight_layout()
save_current_figure('readmitted_distribution.png')
plt.show()

# '<30'이면 1, 'NO' 또는 '>30'이면 0으로 이진화합니다.
eda_df['target_binary'] = (eda_df['readmitted'] == '<30').astype(int)

binary_target_distribution = eda_df['target_binary'].value_counts().sort_index().to_frame('count')
binary_target_distribution['ratio'] = eda_df['target_binary'].value_counts(normalize=True).sort_index()
binary_target_distribution['percent'] = (binary_target_distribution['ratio'] * 100).round(2)
display(binary_target_distribution)

plt.figure(figsize=(5, 4))
sns.countplot(data=eda_df, x='target_binary')
plt.title('Binary Target Distribution')
plt.xlabel('0 = non-<30, 1 = <30')
plt.ylabel('Count')
plt.tight_layout()
save_current_figure('target_binary_distribution.png')
plt.show()

`target_binary`에서 1은 30일 이내 재입원을 의미합니다. 일반적으로 `<30` class는 전체 데이터에서 상대적으로 적게 나타나므로, 이후 모델링 단계에서는 class imbalance를 고려해야 합니다. Logistic Regression과 Random Forest에서는 `class_weight`, XGBoost에서는 `scale_pos_weight` 같은 방법을 사용할 수 있습니다.

## 6. Demographic Variable Analysis

In [ ]:
demographic_vars = ['age', 'gender', 'race']

for col in demographic_vars:
    print(f'\nFrequency table: {col}')
    display(eda_df[col].value_counts(dropna=False).to_frame('count'))

    print(f'\n30-day readmission rate by {col}')
    top_n = 10 if col == 'race' else None
    plot_group_rate(
        eda_df,
        col,
        title=f'30-Day Readmission Rate by {col}',
        filename=f'readmission_rate_by_{col}.png',
        top_n=top_n,
        rotate=True
    )

## 7. Hospital Admission Variable Analysis

In [ ]:
admission_vars = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']

for col in admission_vars:
    print(f'\nFrequency table: {col}')
    display(eda_df[col].value_counts(dropna=False).head(15).to_frame('count'))

    print(f'\n30-day readmission rate by {col}')
    plot_group_rate(
        eda_df,
        col,
        title=f'30-Day Readmission Rate by {col}',
        filename=f'readmission_rate_by_{col}.png',
        top_n=None,
        rotate=True
    )

In [ ]:
print('time_in_hospital summary')
display(eda_df['time_in_hospital'].describe())

plt.figure(figsize=(8, 4))
sns.histplot(data=eda_df, x='time_in_hospital', bins=14, kde=False)
plt.title('Distribution of time_in_hospital')
plt.xlabel('Days in hospital')
plt.ylabel('Count')
plt.tight_layout()
save_current_figure('time_in_hospital_histogram.png')
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=eda_df, x='target_binary', y='time_in_hospital')
plt.title('time_in_hospital by 30-Day Readmission')
plt.xlabel('0 = non-<30, 1 = <30')
plt.ylabel('Days in hospital')
plt.tight_layout()
save_current_figure('time_in_hospital_boxplot_by_target.png')
plt.show()

eda_df['time_in_hospital_group'] = pd.cut(
    eda_df['time_in_hospital'],
    bins=[0, 3, 7, 14],
    labels=['1-3 days', '4-7 days', '8-14 days'],
    include_lowest=True
)

plot_group_rate(
    eda_df,
    'time_in_hospital_group',
    title='30-Day Readmission Rate by Length of Stay Group',
    filename='readmission_rate_by_time_in_hospital_group.png',
    rotate=False
)

## 8. Medical Utilization Variable Analysis

In [ ]:
utilization_vars = ['number_outpatient', 'number_emergency', 'number_inpatient']

for col in utilization_vars:
    print(f'\nSummary: {col}')
    display(eda_df[col].describe())
    print('Top values')
    display(eda_df[col].value_counts().head(15).to_frame('count'))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(data=eda_df, x=col, bins=30, ax=axes[0])
    axes[0].set_title(f'Distribution of {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Count')

    sns.boxplot(data=eda_df, x='target_binary', y=col, ax=axes[1])
    axes[1].set_title(f'{col} by Target')
    axes[1].set_xlabel('0 = non-<30, 1 = <30')
    axes[1].set_ylabel(col)
    plt.tight_layout()
    save_current_figure(f'{col}_distribution_and_boxplot.png')
    plt.show()

# 과거 입원 횟수는 재입원 예측에서 특히 중요할 가능성이 있으므로 구간화해서 확인합니다.
eda_df['number_inpatient_group'] = pd.cut(
    eda_df['number_inpatient'],
    bins=[-1, 0, 1, 2, 5, np.inf],
    labels=['0', '1', '2', '3-5', '6+']
)

plot_group_rate(
    eda_df,
    'number_inpatient_group',
    title='30-Day Readmission Rate by Previous Inpatient Visits',
    filename='readmission_rate_by_number_inpatient_group.png',
    rotate=False
)

## 9. Lab, Medication, and Diagnosis Variable Analysis

In [ ]:
clinical_numeric_vars = ['num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses']

for col in clinical_numeric_vars:
    print(f'\nSummary: {col}')
    display(eda_df[col].describe())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(data=eda_df, x=col, bins=30, ax=axes[0])
    axes[0].set_title(f'Distribution of {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Count')

    sns.boxplot(data=eda_df, x='target_binary', y=col, ax=axes[1])
    axes[1].set_title(f'{col} by 30-Day Readmission')
    axes[1].set_xlabel('0 = non-<30, 1 = <30')
    axes[1].set_ylabel(col)
    plt.tight_layout()
    save_current_figure(f'{col}_distribution_and_boxplot.png')
    plt.show()

In [ ]:
clinical_categorical_vars = ['A1Cresult', 'max_glu_serum', 'insulin', 'diabetesMed', 'change']

for col in clinical_categorical_vars:
    print(f'\nFrequency table: {col}')
    display(eda_df[col].value_counts(dropna=False).to_frame('count'))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.countplot(data=eda_df, x=col, order=eda_df[col].value_counts(dropna=False).index, ax=axes[0])
    axes[0].set_title(f'Distribution of {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)

    rate_table = display_rate_table(eda_df, col)
    sns.barplot(data=rate_table, x=col, y='readmission_rate', ax=axes[1], color='steelblue')
    axes[1].set_title(f'30-Day Readmission Rate by {col}')
    axes[1].set_xlabel(col)
    axes[1].set_ylabel('Readmission rate')
    axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    save_current_figure(f'{col}_distribution_and_rate.png')
    plt.show()

# 진단 코드는 고유값이 많으므로 상위 15개만 확인합니다.
for col in ['diag_1', 'diag_2', 'diag_3']:
    print(f'\nTop diagnosis codes: {col}')
    display(eda_df[col].value_counts(dropna=False).head(15).to_frame('count'))

## 10. Readmission Rate by Groups

In [ ]:
# num_medications도 구간화하여 재입원률을 확인합니다.
eda_df['num_medications_group'] = pd.cut(
    eda_df['num_medications'],
    bins=[0, 10, 20, 30, 50, np.inf],
    labels=['1-10', '11-20', '21-30', '31-50', '51+'],
    include_lowest=True
)

group_analysis_vars = [
    ('age', 'Age'),
    ('gender', 'Gender'),
    ('race', 'Race'),
    ('admission_type_id', 'Admission Type'),
    ('discharge_disposition_id', 'Discharge Disposition'),
    ('insulin', 'Insulin'),
    ('diabetesMed', 'Diabetes Medication'),
    ('number_inpatient_group', 'Previous Inpatient Visit Group'),
    ('num_medications_group', 'Number of Medications Group')
]

group_rate_tables = {}
for col, label in group_analysis_vars:
    print(f'\nReadmission rate by {label}')
    top_n = 10 if col in ['race', 'discharge_disposition_id'] else None
    group_rate_tables[col] = plot_group_rate(
        eda_df,
        col,
        title=f'30-Day Readmission Rate by {label}',
        filename=f'group_rate_{col}.png',
        top_n=top_n,
        rotate=True
    )

## 11. Correlation Analysis

In [ ]:
# 수치형 변수와 target_binary만 선택하여 상관관계를 확인합니다.
numeric_for_corr = eda_df.select_dtypes(include=['int64', 'float64']).copy()

corr_matrix = numeric_for_corr.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Correlation Heatmap for Numeric Variables')
plt.tight_layout()
save_current_figure('numeric_correlation_heatmap.png')
plt.show()

target_corr = corr_matrix['target_binary'].drop('target_binary').sort_values(key=lambda s: s.abs(), ascending=False)
target_corr_df = target_corr.to_frame('correlation_with_target_binary')
display(target_corr_df)

# 변수 간 상관계수가 높은 쌍을 확인합니다.
corr_pairs = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)).stack().reset_index()
corr_pairs.columns = ['feature_1', 'feature_2', 'correlation']
high_corr_pairs = corr_pairs[corr_pairs['correlation'].abs() >= 0.7].sort_values('correlation', key=lambda s: s.abs(), ascending=False)

print('Highly correlated numeric feature pairs with |correlation| >= 0.7')
display(high_corr_pairs)

## 12. Summary of EDA Findings

In [ ]:
# 주요 EDA 결과를 간단히 요약합니다. 값은 notebook 실행 결과에 따라 자동 계산됩니다.
summary = {
    'rows': eda_df.shape[0],
    'columns': eda_df.shape[1],
    'positive_class_count': int((eda_df['target_binary'] == 1).sum()),
    'positive_class_ratio': float((eda_df['target_binary'] == 1).mean()),
    'top_missing_columns': missing_value_summary.head(5).index.tolist(),
    'highest_target_corr_numeric': target_corr_df.head(5).index.tolist()
}

display(pd.Series(summary, name='EDA summary'))

EDA에서 확인할 핵심 내용은 다음과 같습니다.

- 전체 데이터 크기는 약 10만 건의 patient encounter와 여러 임상/입원 관련 변수로 구성되어 있습니다.
- Target인 `<30` class는 `NO`와 `>30`을 합친 non-`<30` class보다 적으므로 class imbalance 가능성이 있습니다.
- `weight`, `payer_code`, `medical_specialty`는 결측 비율이 높아 모델링 단계에서 제거 후보입니다.
- 과거 입원 횟수(`number_inpatient`), 응급 방문 횟수(`number_emergency`), 입원 기간(`time_in_hospital`), 약물 수(`num_medications`), 진단 수(`number_diagnoses`) 등은 재입원 예측에서 중요할 가능성이 있습니다.
- 범주형 변수 중 `age`, `admission_type_id`, `discharge_disposition_id`, `insulin`, `diabetesMed` 등은 그룹별 재입원률 차이를 확인할 필요가 있습니다.
- 전처리에서는 결측치 처리, 범주형 encoding, 식별자 제거, train/test split 이후 preprocessing fit이 중요합니다.

## 13. Preprocessing Recommendations for Modeling

모델링 notebook에서는 다음 전처리 방향을 권장합니다.

- `readmitted`를 binary target으로 변환합니다.
  - `"<30"` -> `1`
  - `"NO"`, `">30"` -> `0`
- `encounter_id`, `patient_nbr`는 식별자 역할을 하므로 제거합니다.
- `?`는 결측치로 처리합니다.
- `weight`, `payer_code`, `medical_specialty`는 결측 비율이 높으므로 제거 후보로 고려합니다.
- 범주형 변수는 One-Hot Encoding을 적용합니다.
- 수치형 변수는 median 등으로 결측치를 처리합니다.
- Logistic Regression과 K-Means에는 `StandardScaler`를 적용합니다.
- Random Forest와 XGBoost는 tree-based model이므로 scaling이 필수는 아닙니다.
- 데이터 누수를 막기 위해 train/test split 이후 preprocessing을 fit합니다.
- Test set에는 train set에서 fit한 preprocessing만 transform합니다.
- 이 EDA notebook에서는 모델 학습, K-Means fitting, scaling, encoding을 수행하지 않고 데이터 이해와 시각화에만 집중합니다.